# 13 — E4: Graph Propagation Depth (RQ3) | SNA-MXH / Group 14

**Research question (proposal):** How does graph propagation depth affect the accuracy–popularity trade-off of LightGCN and RLMRec-Con, and can semantic alignment maintain recommendation accuracy with fewer propagation layers without increasing popularity bias?

**Controlled experiment:** LightGCN (`lightgcn`) vs RLMRec-Con (`lightgcn_plus`), layers ∈ {1,2,3}, seeds ∈ {2023,2024,2025}; RLMRec-Con full alignment, λ=0.01. Amazon Books, unchanged train/validation/test split. **No conclusions before real E4 results.**

**Safety:** Default `DRY_RUN=True`; training requires explicit opt-in. Existing output/checkpoints never overwritten. Uses isolated `/content` workspaces; pinned upstream commit. The 3-layer baselines are *candidates for reuse*, not assumed valid until provenance checks. Analysis requires validated Top-K exports; this notebook does not pretend training logs are evaluation results.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, sys, json, csv, shutil, subprocess, hashlib, time
from datetime import datetime, timezone
import pandas as pd, yaml
ROOT=Path('/content/drive/MyDrive/Data_SNA_Group14')
DATA=ROOT/'data/raw/amazon'
E4=ROOT/'experiments/E4'
SOURCE=Path('/content/RLMRec_E4_source')
REPO_URL='https://github.com/HKUDS/RLMRec.git'
COMMIT='22413752246de3dee8ab0d509f7f7a8889080f95'
MODELS={'LightGCN':'lightgcn','RLMRec-Con':'lightgcn_plus'}
LAYERS=(1,2,3)
SEEDS=(2023,2024,2025)
SEMANTIC_WEIGHT=0.01
DRY_RUN=True            # CHANGE TO False ONLY AFTER PREFLIGHT PASS
SMOKE_TEST_ONLY=True    # True: run just one pending configuration
ALLOW_CLONE=True
RETRY_FAILED=False
assert ROOT.is_dir(), ROOT
print('E4 target:',E4,'mode:', 'DRY RUN' if DRY_RUN else 'TRAINING')


## 1 — Preflight: pin code, validate dataset and configuration

**Important:** `layer_num` is changed *only in an isolated copy* of the repository. E1/E3C files are not modified. We verify that both model YAMLs expose the Amazon graph-layer setting; if not, stop and inspect source before proceeding.

In [ ]:
def git(*args,cwd=None):
    p=subprocess.run(['git',*args],cwd=cwd,text=True,capture_output=True)
    if p.returncode: raise RuntimeError(p.stderr[-2500:])
    return p.stdout.strip()
if not SOURCE.exists():
    if not ALLOW_CLONE: raise FileNotFoundError(SOURCE)
    git('clone',REPO_URL,str(SOURCE))
    git('checkout','--detach',COMMIT,cwd=SOURCE)
assert git('rev-parse','HEAD',cwd=SOURCE)==COMMIT, 'Repository commit differs: do not silently switch.'
assert not git('status','--porcelain',cwd=SOURCE), 'Source is dirty; use a clean runtime.'
required=['trn_mat.pkl','val_mat.pkl','tst_mat.pkl','usr_emb_np.pkl','itm_emb_np.pkl']
assert all((DATA/f).is_file() for f in required), [f for f in required if not (DATA/f).is_file()]
link=SOURCE/'data/amazon'; link.parent.mkdir(parents=True,exist_ok=True)
if not link.exists() and not link.is_symlink(): link.symlink_to(DATA,target_is_directory=True)
assert link.resolve()==DATA.resolve(), 'Dataset path is not the approved Drive split.'
BASE_CONFIG={}
for label,model in MODELS.items():
    path=SOURCE/f'encoder/config/modelconf/{model}.yml'
    assert path.is_file(), f'Missing config: {path}'
    cfg=yaml.safe_load(path.read_text())
    assert 'model' in cfg and 'train' in cfg
    assert 'amazon' in cfg['model'] and 'layer_num' in cfg['model']['amazon'], f'Inspect layer key for {model}'
    assert int(cfg['model']['amazon']['layer_num'])==3, f'Unexpected E1 baseline layer count: {model}'
    if model=='lightgcn_plus':
        assert abs(float(cfg['model']['amazon']['kd_weight'])-SEMANTIC_WEIGHT)<1e-12
    BASE_CONFIG[model]=cfg
print('PREFLIGHT PASS: pinned code, dataset, 3-layer reference, model config.')
try:
    import torch
    print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
except ImportError: print('PyTorch not installed')


## 2 — Manifest: 18 controlled model/layer/seed cells

Do **not** assume a baseline can be reused merely because its model name matches. For 3-layer runs, E1 candidates are listed for manual verification (split, seed, configuration, checkpoint, Top-K provenance). This notebook initially treats them as pending unless a verified E4 run already exists.

In [ ]:
def run_dir(model,layer,seed): return E4/'runs'/model/f'layers_{layer}'/f'seed_{seed}'
def state(model,layer,seed):
    d=run_dir(model,layer,seed)
    p=d/'train_status.json'
    if p.exists():
        x=json.loads(p.read_text())
        return x.get('status','unknown')
    return 'pending' if not d.exists() else 'unverified_existing_directory'
rows=[]
for label,model in MODELS.items():
    for layer in LAYERS:
        for seed in SEEDS:
            rows.append({'label':label,'model':model,'layers':layer,'seed':seed,'status':state(model,layer,seed),
                         'baseline_candidate':layer==3,
                         'output':str(run_dir(model,layer,seed))})
manifest=pd.DataFrame(rows)
display(manifest[['label','layers','seed','status','baseline_candidate']])
print('Planned combinations:',len(manifest),'| pending:',sum(manifest.status=='pending'))


## 3 — Execute training in isolated workspaces (explicit opt-in)

This follows the **E3C notebook's pinned-repo / isolated-workspace / CLI training pattern**, but changes only `layer_num` and seed; RLMRec-Con semantic weight stays 0.01. Run one smoke test before the full sweep. No deletion or overwrite of old results. Failed runs require manual review.

In [ ]:
def write_json(p,obj):
    p.write_text(json.dumps(obj,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
def run_one(model,layer,seed):
    import torch
    assert torch.cuda.is_available(), 'Enable GPU runtime.'
    assert model in MODELS.values() and layer in LAYERS and seed in SEEDS
    dest=run_dir(model,layer,seed)
    assert state(model,layer,seed)=='pending' or (RETRY_FAILED and state(model,layer,seed)=='failed'), f'Existing run must be reviewed: {dest}'
    if dest.exists():
        raise RuntimeError('Refusing to overwrite existing run: '+str(dest))
    work=Path('/content')/f'e4_work_{model}_L{layer}_s{seed}'
    assert not work.exists(), f'Workspace exists: {work}'
    def ignore(src,names):
        return set(names)&{'.git','data','checkpoint','checkpoints','log','logs','__pycache__','.ipynb_checkpoints'}
    shutil.copytree(SOURCE,work,ignore=ignore,symlinks=True)
    (work/'data').symlink_to(SOURCE/'data',target_is_directory=True)
    config_file=work/f'encoder/config/modelconf/{model}.yml'
    cfg=yaml.safe_load(config_file.read_text())
    cfg['model']['amazon']['layer_num']=int(layer)
    if 'layer_num' in cfg['model']: cfg['model']['layer_num']=int(layer)
    if model=='lightgcn_plus':
        cfg['model']['amazon']['kd_weight']=SEMANTIC_WEIGHT
        cfg['model']['kd_weight']=SEMANTIC_WEIGHT
    cfg['train']['seed']=int(seed)
    cfg['train']['save_model']=True
    config_file.write_text(yaml.safe_dump(cfg,sort_keys=False),encoding='utf-8')
    dest.mkdir(parents=True,exist_ok=False)
    (dest/'config_snapshot.yml').write_text(config_file.read_text())
    meta={'experiment':'E4_Graph_Depth','model':model,'layers':layer,'seed':seed,
          'source_commit':COMMIT,'semantic_weight':SEMANTIC_WEIGHT if model=='lightgcn_plus' else None,
          'status':'running','started_utc':datetime.now(timezone.utc).isoformat()}
    write_json(dest/'train_status.json',meta)
    cmd=[sys.executable,'encoder/train_encoder.py','--model',model,'--dataset','amazon',
         '--device','cuda','--cuda','0','--seed',str(seed)]
    try:
        with (dest/'training_console.log').open('w',encoding='utf-8') as f:
            proc=subprocess.Popen(cmd,cwd=work,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace',bufsize=1)
            for line in proc.stdout:
                f.write(line);f.flush()
                if any(x in line for x in ('Epoch','Recall','NDCG','Error','Traceback','Best')): print(line.rstrip(),flush=True)
            rc=proc.wait()
        meta['exit_code']=rc
        if rc: raise RuntimeError(f'Exit {rc}; inspect training_console.log')
        artifacts=[]
        for folder in ('checkpoint','checkpoints','log','logs','save','saved'):
            src=work/folder
            if src.is_dir():
                shutil.copytree(src,dest/folder)
                artifacts.append(folder)
        checkpoints=list(dest.rglob('*.pth'))+list(dest.rglob('*.pt'))
        meta['checkpoint_count']=len(checkpoints)
        meta['artifacts']=artifacts
        meta['status']='trained_pending_export' if checkpoints else 'training_finished_check_checkpoint'
    except Exception as exc:
        meta['status']='failed';meta['error']=repr(exc)
        raise
    finally:
        meta['finished_utc']=datetime.now(timezone.utc).isoformat()
        write_json(dest/'train_status.json',meta)
        # Keep failed workspace for debugging; do not automatically delete anything.
        if meta['status']=='trained_pending_export':
            shutil.rmtree(work)
    return meta


In [ ]:
# EXECUTION GATE — set DRY_RUN=False in cell 2 after reviewing the manifest.
manifest=pd.DataFrame([{'model':m,'layers':l,'seed':s,'status':state(m,l,s)}
                       for m in MODELS.values() for l in LAYERS for s in SEEDS])
if DRY_RUN:
    print('SAFE DRY RUN: no training. Set DRY_RUN=False and rerun from config cell when ready.')
else:
    pending=[(r.model,int(r.layers),int(r.seed)) for r in manifest.itertuples() if r.status=='pending']
    if SMOKE_TEST_ONLY: pending=pending[:1]
    print('Training queue:',pending)
    for m,l,s in pending:
        print(run_one(m,l,s))


## 4 — Verification / RQ3 analysis handoff

**Do not infer RQ3 from training loss or E3C weight sweep.** Each E4 configuration needs a **verified train-masked Top-K export** (K=20) and the same evaluation protocol as E2/E3C. Verify source checkpoint, split, test user set, no train interactions in Top-K, no duplicates, and source provenance. Then compute Recall/NDCG, ARP, APLT, conditional Head/Tail Recall, coverage and degree–exposure Spearman at K=10/20. The head set is top ceil(20%) by *train-only* item degree, tie-break by item ID.

Primary RQ3 comparisons:
1. **Within model:** depth 1 → 2 → 3 (accuracy and popularity jointly).
2. **Across models at fixed depth:** RLMRec-Con vs LightGCN.
3. **Fewer-layer test:** RLMRec-Con at depth 1/2 vs LightGCN at depth 3. A lower-layer configuration qualifies as favorable only if its accuracy is competitive **and** popularity bias does not worsen under prespecified metrics.
4. Mean ± sample SD across three matched seeds, plus seed-paired differences. Do not claim significance from n=3.

**Checkpoint reuse rule:** 3-layer E1 runs may be reused only after verifying exact source commit, model configuration, data split, seed, and checkpoint/export provenance. Otherwise run them explicitly. The `E3C` analysis is not an E4 substitute.


In [ ]:
# Read-only readiness report. This is NOT an analysis of RQ3 until Top-K QA and metrics exist.
records=[]
for label,model in MODELS.items():
    for layer in LAYERS:
        for seed in SEEDS:
            d=run_dir(model,layer,seed)
            topk=d/'topk/topk_test.npz'
            records.append({'model':label,'layers':layer,'seed':seed,'training_status':state(model,layer,seed),
                            'topk_exists':topk.is_file(), 'topk_path':str(topk)})
readiness=pd.DataFrame(records)
display(readiness)
if readiness.topk_exists.all():
    print('All 18 Top-K files exist; NEXT: run independent E4 QA + E2/E3C metric evaluator before drawing conclusions.')
else:
    print('E4 NOT YET ANALYZABLE: missing',sum(~readiness.topk_exists),'Top-K exports (unless approved 3-layer reuse is separately documented).')
